In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob
import zipfile
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 33
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## Pew GRI Pipeline

**Source:** Pew Research Center — Global Restrictions on Religion
**Access:** Manual ZIP download (requires free Pew account)
**Download instructions:** See `docs/instructions_data_maintenance.md` — PEW_GRI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Government Restrictions Index (GRI) | Civil liberties / religious freedom | Primary tier 2 |
| Social Hostilities Index (SHI) | Civil liberties / religious freedom | Primary tier 2 |

In [3]:
import pandas as pd
import os
import glob
import zipfile
import io
from datetime import datetime

# Auto-detect Pew GRI ZIP in Downloads — no hardcoded filename
gri_pattern = os.path.join(DOWNLOADS_DIR, "Global-Restrictions-on-Religion*.zip")
gri_files = glob.glob(gri_pattern)

if not gri_files:
    print(f"No Pew GRI ZIP found in {DOWNLOADS_DIR}")
    print("Download from pewresearch.org/religion-datasets/ (search Global Restrictions on Religion)")
else:
    # Use most recent download
    gri_zip = max(gri_files, key=os.path.getmtime)
    print(f"Found: {os.path.basename(gri_zip)}")
    
    # Inspect ZIP contents
    with zipfile.ZipFile(gri_zip) as z:
        print(f"Contents:")
        for name in z.namelist():
            print(f"  {name}")

Found: Global-Restrictions-on-Religion-2007-2022-Dataset.zip
Contents:
  PublicDataset_ReligiousRestrictions_2007to2022.csv
  __MACOSX/._PublicDataset_ReligiousRestrictions_2007to2022.csv
  PublicDataset_ReligiousRestrictions_2007to2022.dta
  __MACOSX/._PublicDataset_ReligiousRestrictions_2007to2022.dta
  README.txt
  __MACOSX/._README.txt
  Religious Restrictions Codebook.pdf
  __MACOSX/._Religious Restrictions Codebook.pdf
  Religious Restrictions Dataset Note.txt
  __MACOSX/._Religious Restrictions Dataset Note.txt


In [4]:
# Load the CSV directly from the ZIP — read the main file, ignore __MACOSX entries
with zipfile.ZipFile(gri_zip) as z:
    csv_name = [n for n in z.namelist() 
                if n.endswith('.csv') and not n.startswith('__MACOSX')][0]
    print(f"Loading: {csv_name}")
    with z.open(csv_name) as f:
        gri_raw = pd.read_csv(f)

print(f"\nShape: {gri_raw.shape}")
print(f"Columns: {list(gri_raw.columns)}")
print(gri_raw.head(3))

Loading: PublicDataset_ReligiousRestrictions_2007to2022.csv

Shape: (3164, 81)
Columns: ['Nation_fk', 'Ctry_EditorialName', 'Region5', 'Question_Year', 'GRI', 'SHI', 'GRI_Q_1', 'GRI_Q_2', 'GRI_Q_3', 'GRI_Q_4', 'GRI_Q_5', 'GRI_Q_6', 'GRI_Q_7', 'GRI_Q_8', 'GRI_Q_9', 'GRI_Q_10', 'GRI_Q_11', 'GRI_Q_11_Unaffiliated', 'GRI_Q_11_Christianity', 'GRI_Q_11_Islam', 'GRI_Q_11_Buddhism', 'GRI_Q_11_Hinduism', 'GRI_Q_11_Judaism', 'GRI_Q_11_Other_Religions', 'GRI_Q_11_Folk_Religions', 'GRI_Q_12', 'GRI_Q_13', 'GRI_Q_14', 'GRI_Q_15', 'GRI_Q_16_Reasons', 'GRI_Q_16', 'GRI_Q_17', 'GRI_Q_18', 'GRI_Q_19', 'GRI_Q_19_Extent', 'GRI_Q_19_Property_Damage', 'GRI_Q_19_Detentions', 'GRI_Q_19_Displacements', 'GRI_Q_19_Abuse', 'GRI_Q_19_Deaths', 'GRI_Q_20_1', 'GRI_Q_20_2', 'GRI_Q_20_3_a', 'GRI_Q_20_3_b', 'GRI_Q_20_3_c', 'GRI_Q_20_3', 'GRI_Q_20_4', 'GRI_Q_20_5', 'GRX_22_Blasphemy', 'GRX_22_Apostasy', 'GRX_22_Hate_Speech', 'GRX_22_Criticism_of_Religion', 'SHI_Q_1_Extent', 'SHI_Q_1_Harassment', 'SHI_Q_1_Property_Damage',

In [5]:
# Select the two headline indices plus identifiers
# GRI = Government Restrictions Index (0-10), SHI = Social Hostilities Index (0-10)
KEEP_COLS = {
    'Ctry_EditorialName': 'country_name',
    'Question_Year':      'year',
    'GRI':                'pew_gov_restrictions_index',
    'SHI':                'pew_social_hostilities_index',
}

gri = gri_raw[list(KEEP_COLS.keys())].copy()
gri = gri.rename(columns=KEEP_COLS)

# Filter to framework start year
gri = gri[gri['year'] >= FRAMEWORK_START_YEAR].copy()
gri = gri.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {gri.shape}")
print(f"Years: {gri['year'].min()} — {gri['year'].max()}")
print(f"Countries: {gri['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (gri.isnull().sum() / len(gri) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(f"\nGRI range: {gri['pew_gov_restrictions_index'].min()} — {gri['pew_gov_restrictions_index'].max()}")
print(f"SHI range: {gri['pew_social_hostilities_index'].min()} — {gri['pew_social_hostilities_index'].max()}")

Shape: (3164, 4)
Years: 2007 — 2022
Countries: 198

Missing values (%):
Series([], dtype: float64)

GRI range: 0.0 — 9.335
SHI range: 0.0 — 10.0


In [6]:
# Derive metadata from data — no hardcoding
latest_year = str(int(gri['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "pew_gri_clean.csv")
gri.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {gri.shape}")

# Update download log
update_entry(
    "PEW_GRI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="pew_gri_clean.csv",
    latest_available_version=latest_year,
    notes="Pew Government Restrictions Index (GRI) and Social Hostilities Index (SHI), both 0-10. "
          "Manual ZIP download (free Pew account). Pipeline auto-detects ZIP and extracts CSV. "
          "Headline indices only — 79 question-level columns dropped. "
          "Coverage: 198 countries. Annual."
)
print_entry("PEW_GRI")

Written: C:\Users\mjbou\governance-framework\data\processed\pew_gri_clean.csv
Shape: (3164, 4)
[download_log] Updated entry for PEW_GRI
  source_id: PEW_GRI
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2022
  local_filename: pew_gri_clean.csv
  latest_available_version: 2022
  no_update_reason: nan
  notes: Pew Government Restrictions Index (GRI) and Social Hostilities Index (SHI), both 0-10. Manual ZIP download (free Pew account). Pipeline auto-detects ZIP and extracts CSV. Headline indices only — 79 question-level columns dropped. Coverage: 198 countries. Annual.
